In [ ]:
This notebook loads the existing report data, identifies missing `variance`, `median`, and `count` values, fills them, and summarizes the final insight.</VSCode.Cell>
<VSCode.Cell language="python">import json
from pathlib import Path
import pandas as pd
from docx import Document

REPORT_DIR = Path(r'e:/tasnim/Graduation project model/tasnim/evaluation/statistical_analysis/reports')
FULL_REPORT_PATH = REPORT_DIR / 'evaluation_full_report.docx'
STAT_REPORT_PATH = REPORT_DIR / 'statistical_analysis_report.docx'
print('Report directory:', REPORT_DIR)
print('Full report exists:', FULL_REPORT_PATH.exists())
print('Statistical report exists:', STAT_REPORT_PATH.exists())</VSCode.Cell>
<VSCode.Cell language="markdown">## Load the Tables into DataFrames
Read the report tables from the Word documents and represent them as pandas DataFrames for inspection.</VSCode.Cell>
<VSCode.Cell language="python">def docx_tables_to_dataframes(doc_path: Path):
    doc = Document(str(doc_path))
    tables = []
    for table in doc.tables:
        rows = []
        headers = [cell.text.strip() for cell in table.rows[0].cells]
        for row in table.rows[1:]:
            row_data = {headers[i]: row.cells[i].text.strip() for i in range(len(headers))}
            rows.append(row_data)
        tables.append((headers, pd.DataFrame(rows)))
    return tables

full_tables = docx_tables_to_dataframes(FULL_REPORT_PATH)
stat_tables = docx_tables_to_dataframes(STAT_REPORT_PATH)
print('Full report table count:', len(full_tables))
print('Stat report table count:', len(stat_tables))
for i, (headers, df) in enumerate(stat_tables, 1):
    print('\nStat report table', i)
    print('Headers:', headers)
    print(df.head(3).to_dict(orient='records'))</VSCode.Cell>
<VSCode.Cell language="markdown">## Inspect Missing Columns and Data Types
Check which tables contain empty `variance`, `median`, or `count` values and confirm the relevant numeric columns.</VSCode.Cell>
<VSCode.Cell language="python">def find_missing_statistics(df: pd.DataFrame):
    missing = {}
    for col in ['Variance', 'Median', 'Count', 'variance', 'median', 'count']:
        if col in df.columns:
            missing[col] = df[col].isna() | (df[col].astype(str).str.strip() == '')
    return missing

for i, (headers, df) in enumerate(stat_tables, 1):
    missing = find_missing_statistics(df)
    if any(missing.values()):
        print('\nTable', i, 'missing columns:')
        for col, mask in missing.items():
            if mask.any():
                print(f' {col}: {mask.sum()} missing')
    else:
        print('\nTable', i, 'has no missing statistic columns')
    print(df.dtypes)
</VSCode.Cell>
